In [38]:
SIMULATION_DATE = "2026-03-19"

from config import r, STOCKS, FILEPATH, s3, EXCHANGE
import os

def download_file():
    local_path = f"{FILEPATH}/{EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}.csv"
    if not os.path.exists(local_path):
        
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        try:
            s3.download_file(
                Bucket="cashcow",
                Key=f"{EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}.csv",
                Filename=local_path
            )
            print(f"[MAIN] Downloaded {EXCHANGE}:{STOCKS[0]} for {SIMULATION_DATE}")
        except Exception as e:
            raise Exception(f"file not found, choose another date. key={EXCHANGE}/{STOCKS[0]}/{SIMULATION_DATE}.csv") from e
            

download_file()

In [39]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from simulator import get_all_tick_data
df = get_all_tick_data(SIMULATION_DATE)

from StockAnalyser import Delta_analysis
instance = Delta_analysis()
for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing ticks"):
    instance.parse(row.to_dict())

✅ Cache is up to date, no refresh needed
[INFO] loaded data into memory for simulation


Parsing ticks: 100%|██████████| 27272/27272 [00:02<00:00, 11464.12it/s]


In [37]:
import numpy as np
import pyqtgraph as pg
from PyQt5.QtCore import QRectF

def get_active_price_bounds_local(inst):
    active = np.where((inst.aggdf_buy > 0) & (inst.aggdf_sell > 0))[0]
    if active.size == 0:
        return 0, max(0, inst.WIDTH - 1)
    return int(active[0]), int(active[-1])

# ---------- Build heatmap data from Delta_analysis numpy arrays ----------
rows_used = int(instance.curr_time_idx)
if rows_used <= 0:
    raise ValueError("No parsed ticks available in instance.")

min_idx, max_idx = get_active_price_bounds_local(instance)
col_slice = slice(min_idx, max_idx + 1)

# 1. Extract matrices
lh_buy = instance.lowHigh['buy'][:rows_used, col_slice]
hl_buy = instance.highLow['buy'][:rows_used, col_slice]
lh_sell = instance.lowHigh['sell'][:rows_used, col_slice]
hl_sell = instance.highLow['sell'][:rows_used, col_slice]

# 2. Find where data is valid (Not NaN)
valid_buy = ~np.isnan(lh_buy)
valid_sell = ~np.isnan(lh_sell)

# 3. Calculate codes cleanly as integers
# 3. Calculate codes cleanly as integers (Initialize to 3 to match Pandas NaN mapping!)
buy_code = np.zeros_like(lh_buy, dtype=np.uint8)
sell_code = np.zeros_like(lh_sell, dtype=np.uint8)

buy_code[valid_buy] = (
    (lh_buy[valid_buy] >= 0).astype(np.uint8)
    + 2 * (hl_buy[valid_buy] >= 0).astype(np.uint8)
)

sell_code[valid_sell] = (
    (lh_sell[valid_sell] >= 0).astype(np.uint8)
    + 2 * (hl_sell[valid_sell] >= 0).astype(np.uint8)
)# 4. Combine into strict integers (0 to 15)
heatmap_data = sell_code + 4 * buy_code

# 5. Replicate Pandas NaN propagation: If EITHER buy or sell is missing, the pixel is transparent
invalid = ~(valid_buy & valid_sell)
heatmap_data[invalid] = 16  # 16 is our Transparent color

# ---------- PyQtGraph setup ----------
app = pg.mkQApp("Delta Analysis Heatmap")
win = pg.GraphicsLayoutWidget(title="Delta_analysis Heatmap & LTP")
win.setBackground("white")
plot = win.addPlot(title="Combined State (0..15)")
plot.showGrid(x=True, y=True, alpha=0.15)
win.show()

# 17 Base Colors
base_lut = np.array([
    [255, 0, 0, 170],       # 0: Red
    [0, 114, 178, 170],     # 1: Blue
    [230, 159, 0, 170],     # 2: Orange
    [86, 180, 233, 170],    # 3: Sky Blue
    [0, 158, 115, 170],     # 4: Bluish Green
    [204, 121, 167, 170],   # 5: Purple
    [240, 228, 66, 170],    # 6: Yellow
    [0, 0, 128, 170],       # 7: Navy
    [255, 105, 180, 170],   # 8: Pink
    [27, 94, 32, 170],      # 9: Dark Green
    [139, 69, 19, 170],     # 10: Brown

    [128, 0, 0, 170],       # 11: Maroon
    [0, 128, 128, 170],     # 12: Teal
    [245, 245, 245, 170],   # 13: Off-White
    [119, 119, 119, 170],   # 14: Gray
    [57, 255, 20, 170],     # 15: Neon Green
    [0, 0, 0, 0],           # 16: TRANSPARENT (Background)
], dtype=np.ubyte)

# THE MAGIC TRICK: Pad the LUT to exactly 256 rows so PyQtGraph does direct indexing!
full_lut = np.zeros((256, 4), dtype=np.ubyte)
full_lut[:17] = base_lut

img = pg.ImageItem()
img.setLookupTable(full_lut)

# Passing the uint8 array to a 256-color LUT guarantees pixel-perfect mapping.
img.setImage(heatmap_data, autoLevels=False)
plot.addItem(img)

min_ltp = float(instance.base_ltp + min_idx)
max_ltp = float(instance.base_ltp + max_idx)
img.setRect(QRectF(0, min_ltp, rows_used, max(1.0, max_ltp - min_ltp + 1.0)))

ltp_values = instance.ltpdf[:rows_used, 0].astype(float)
valid_ltp = np.isfinite(ltp_values) & (ltp_values != 0)

x = np.arange(rows_used, dtype=float)[valid_ltp]
y = ltp_values[valid_ltp]
plot.plot(x, y, pen=pg.mkPen(color=(0, 0, 255), width=2), name='LTP')

plot.setLabel('bottom', 'Tick Index')
plot.setLabel('left', 'LTP')
plot.setYRange(min_ltp, max_ltp, padding=0)

app.exec()

0

In [19]:
import pandas as pd
pd.DataFrame(heatmap_data).to_csv("heatmap_data.csv", index=False, header=False)

In [18]:
heatmap_data.T.shape

(418, 10659)

In [34]:
import numpy as np
import pyqtgraph as pg
from PyQt5.QtCore import QRectF

def get_active_price_bounds_local(inst):
    active = np.where((inst.aggdf_buy > 0) | (inst.aggdf_sell > 0))[0]
    if active.size == 0:
        return 0, max(0, inst.WIDTH - 1)
    return int(active[0]), int(active[-1])

rows_used = int(instance.curr_time_idx)
if rows_used <= 0:
    raise ValueError("No parsed ticks available in instance.")

min_idx, max_idx = get_active_price_bounds_local(instance)
col_slice = slice(min_idx, max_idx + 1)

# 1. Extract the raw numpy matrices
lh_buy = instance.lowHigh['buy'][:rows_used, col_slice]
hl_buy = instance.highLow['buy'][:rows_used, col_slice]
lh_sell = instance.lowHigh['sell'][:rows_used, col_slice]
hl_sell = instance.highLow['sell'][:rows_used, col_slice]

# 2. THE PANDAS CLONE: ~(x < 0) converts Negatives to 0, and Positives/NaNs to 1!
# This perfectly mimics your old `lambda x: 0 if x < 0 else 1`
buy_code = (~(lh_buy < 0)).astype(float) + 2.0 * (~(hl_buy < 0)).astype(float)
sell_code = (~(lh_sell < 0)).astype(float) + 2.0 * (~(hl_sell < 0)).astype(float)

# Combine into the 0-15 heatmap data
heatmap_data = sell_code + 4.0 * buy_code

# 3. Transparent Background: If BOTH sides are NaN, it means the price hasn't traded yet.
# We set it back to NaN so PyQtGraph makes it white/transparent.
untraded_background = np.isnan(lh_buy) & np.isnan(lh_sell)
heatmap_data[untraded_background] = np.nan

# ---------- PyQtGraph setup ----------
app = pg.mkQApp("Delta Analysis Heatmap")
win = pg.GraphicsLayoutWidget(title="Delta_analysis Heatmap & LTP")
win.setBackground("white")
plot = win.addPlot(title="Combined State (0..15)")
plot.showGrid(x=True, y=True, alpha=0.15)
win.show()

# YOUR ORIGINAL LUT IS BACK
lut = np.array([
    [255,   0,   0, 170],   #  0: Red           (Sell: Neg/Neg | Buy: Neg/Neg)
    [230, 159,   0, 170],   #  1: Orange        (Sell: Pos/Neg | Buy: Neg/Neg)
    [  0, 114, 178, 170],   #  2: Blue          (Sell: Neg/Pos | Buy: Neg/Neg)
    [ 86, 180, 233, 170],   #  3: Sky Blue      (Sell: Pos/Pos | Buy: Neg/Neg)
    [255, 105, 180, 170],   #  4: Pink          (Sell: Neg/Neg | Buy: Pos/Neg)
    [  0,   0, 128, 170],   #  5: Navy          (Sell: Pos/Neg | Buy: Pos/Neg)
    [ 27,  94,  32, 170],   #  6: Dark Green    (Sell: Neg/Pos | Buy: Pos/Neg)
    [128,   0,   0, 170],   #  7: Maroon        (Sell: Pos/Pos | Buy: Pos/Neg)
    [  0, 158, 115, 170],   #  8: Bluish Green  (Sell: Neg/Neg | Buy: Neg/Pos)
    [240, 228,  66, 170],   #  9: Yellow        (Sell: Pos/Neg | Buy: Neg/Pos)
    [204, 121, 167, 170],   # 10: Purple        (Sell: Neg/Pos | Buy: Neg/Pos)
    [139,  69,  19, 170],   # 11: Brown         (Sell: Pos/Pos | Buy: Neg/Pos)
    [  0, 128, 128, 170],   # 12: Teal          (Sell: Neg/Neg | Buy: Pos/Pos)
    [119, 119, 119, 170],   # 13: Gray          (Sell: Pos/Neg | Buy: Pos/Pos)
    [245, 245, 245, 170],   # 14: Off-White     (Sell: Neg/Pos | Buy: Pos/Pos)
    [ 57, 255,  20, 170],   # 15: Neon Green    (Sell: Pos/Pos | Buy: Pos/Pos)
], dtype=np.ubyte)

img = pg.ImageItem()
img.setLookupTable(lut)

# Lock levels so 0 is strictly Red, and 15 is Neon Green
img.setLevels([0, 15]) 

# Pass the raw float matrix so NaNs render transparent!
img.setImage(heatmap_data, autoLevels=False)
plot.addItem(img)

# Set the bounding box
min_ltp = float(instance.base_ltp + min_idx)
max_ltp = float(instance.base_ltp + max_idx)
img.setRect(QRectF(0, min_ltp, rows_used, max(1.0, max_ltp - min_ltp + 1.0)))

# Add the LTP line
ltp_values = instance.ltpdf[:rows_used, 0].astype(float)
valid_ltp = np.isfinite(ltp_values) & (ltp_values != 0)

x = np.arange(rows_used, dtype=float)[valid_ltp]
y = ltp_values[valid_ltp]
plot.plot(x, y, pen=pg.mkPen(color=(0, 0, 255), width=2), name='LTP')


plot.setLabel('bottom', 'Tick Index')
plot.setLabel('left', 'LTP')
plot.setYRange(min_ltp, max_ltp, padding=0)

app.exec()

0

In [35]:
import numpy as np
import pyqtgraph as pg
from PyQt5.QtCore import QRectF

def get_active_price_bounds_local(inst):
    active = np.where((inst.aggdf_buy > 0) | (inst.aggdf_sell > 0))[0]
    if active.size == 0:
        return 0, max(0, inst.WIDTH - 1)
    return int(active[0]), int(active[-1])

rows_used = int(instance.curr_time_idx)
if rows_used <= 0:
    raise ValueError("No parsed ticks available in instance.")

min_idx, max_idx = get_active_price_bounds_local(instance)
col_slice = slice(min_idx, max_idx + 1)

# 1. Extract the raw numpy matrices
lh_buy = instance.lowHigh['buy'][:rows_used, col_slice]
hl_buy = instance.highLow['buy'][:rows_used, col_slice]
lh_sell = instance.lowHigh['sell'][:rows_used, col_slice]
hl_sell = instance.highLow['sell'][:rows_used, col_slice]

# 2. Masks for where data is actually traded
valid_buy = ~np.isnan(lh_buy)
valid_sell = ~np.isnan(lh_sell)

# 3. Initialize to 3 to perfectly mimic Pandas lambda (`NaN` evaluating to 1 -> 1 + 2(1) = 3)
buy_code = np.full_like(lh_buy, 3, dtype=np.uint8)
sell_code = np.full_like(lh_sell, 3, dtype=np.uint8)

# 4. Evaluate the true math ONLY where data exists
buy_code[valid_buy] = (lh_buy[valid_buy] >= 0).astype(np.uint8) + 2 * (hl_buy[valid_buy] >= 0).astype(np.uint8)
sell_code[valid_sell] = (lh_sell[valid_sell] >= 0).astype(np.uint8) + 2 * (hl_sell[valid_sell] >= 0).astype(np.uint8)

# Combine into the 0-15 heatmap states
heatmap_data = sell_code + 4 * buy_code

# 5. Handle the Global Untraded Background (The White Area)
untraded = np.isnan(lh_buy) & np.isnan(lh_sell)
heatmap_data[untraded] = 16  # 16 is our custom Transparent color

# ---------- PyQtGraph setup ----------
app = pg.mkQApp("Delta Analysis Heatmap")
win = pg.GraphicsLayoutWidget(title="Delta_analysis Heatmap & LTP")
win.setBackground("white")
plot = win.addPlot(title="Combined State (0..15)")
plot.showGrid(x=True, y=True, alpha=0.15)
win.show()

# THE PERFECTLY MAPPED LUT (Derived exactly from your uploaded image)
base_lut = np.array([
    [255,   0,   0, 170],   #  0: Red           (Sell: Neg/Neg | Buy: Neg/Neg)
    [230, 159,   0, 170],   #  1: Orange        (Sell: Pos/Neg | Buy: Neg/Neg)
    [  0, 114, 178, 170],   #  2: Blue          (Sell: Neg/Pos | Buy: Neg/Neg)
    [ 86, 180, 233, 170],   #  3: Sky Blue      (Sell: Pos/Pos | Buy: Neg/Neg)
    [255, 105, 180, 170],   #  4: Pink          (Sell: Neg/Neg | Buy: Pos/Neg)
    [  0,   0, 128, 170],   #  5: Navy          (Sell: Pos/Neg | Buy: Pos/Neg)
    [ 27,  94,  32, 170],   #  6: Dark Green    (Sell: Neg/Pos | Buy: Pos/Neg)
    [128,   0,   0, 170],   #  7: Maroon        (Sell: Pos/Pos | Buy: Pos/Neg)
    [  0, 158, 115, 170],   #  8: Bluish Green  (Sell: Neg/Neg | Buy: Neg/Pos)
    [240, 228,  66, 170],   #  9: Yellow        (Sell: Pos/Neg | Buy: Neg/Pos)
    [204, 121, 167, 170],   # 10: Purple        (Sell: Neg/Pos | Buy: Neg/Pos)
    [139,  69,  19, 170],   # 11: Brown         (Sell: Pos/Pos | Buy: Neg/Pos)
    [  0, 128, 128, 170],   # 12: Teal          (Sell: Neg/Neg | Buy: Pos/Pos)
    [119, 119, 119, 170],   # 13: Gray          (Sell: Pos/Neg | Buy: Pos/Pos)
    [245, 245, 245, 170],   # 14: Off-White     (Sell: Neg/Pos | Buy: Pos/Pos)
    [ 57, 255,  20, 170],   # 15: Neon Green    (Sell: Pos/Pos | Buy: Pos/Pos)
], dtype=np.ubyte)

# THE MAGIC TRICK: Pad the LUT to exactly 256 rows to force direct bitmap rendering
full_lut = np.zeros((256, 4), dtype=np.ubyte)
full_lut[:16] = base_lut
full_lut[16] = [0, 0, 0, 0] # Transparent for the background!

img = pg.ImageItem()
img.setLookupTable(full_lut)

# Passing the uint8 array with the padded LUT guarantees 100% pixel perfection
img.setImage(heatmap_data, autoLevels=False)
plot.addItem(img)

# Set the bounding box
min_ltp = float(instance.base_ltp + min_idx)
max_ltp = float(instance.base_ltp + max_idx)
img.setRect(QRectF(0, min_ltp, rows_used, max(1.0, max_ltp - min_ltp + 1.0)))

# Add the LTP line
ltp_values = instance.ltpdf[:rows_used, 0].astype(float)
valid_ltp = np.isfinite(ltp_values) & (ltp_values != 0)

x = np.arange(rows_used, dtype=float)[valid_ltp]
y = ltp_values[valid_ltp]
plot.plot(x, y, pen=pg.mkPen(color=(0, 0, 255), width=2), name='LTP')

plot.setLabel('bottom', 'Tick Index')
plot.setLabel('left', 'LTP')
plot.setYRange(min_ltp, max_ltp, padding=0)

app.exec()

0